<font size=10>**CO-BIDDING NETWORK INFERENCE**</font>

**Following Sturm et al. (2025), Section 4**

<font color='#BFD72F' size=5>**METHOD**:</font> Centered Jaccard-Tanimoto Coefficient with Statistical Significance Testing

*"Each node represents an active firm and each edge connects a pair of firms if their co-bidding behavior is statistically significant."*

<font color='#BFD72F' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>
- [1. Imports & Data Loading](#1-imports)
- [2. Binary Bid Matrix](#2-bid-matrix)
- [3. Centered Jaccard-Tanimoto Coefficient](#3-jaccard)
- [4. Statistical Significance Testing](#4-significance)
- [5. Network Construction](#5-network)
- [6. Summary Statistics](#6-summary)
- [7. Export](#7-export)

# <font color='#BFD72F' size=6>**1. Imports & Data Loading**</font> <a class="anchor" id="1-imports"></a>
[Back to TOC](#toc)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse
from scipy import stats
from itertools import combinations
import warnings
import os
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

print('Libraries loaded.')

In [ ]:
# Load preprocessed data from NB01
bids_table = pd.read_csv('../data/bids_table.csv')
contracts = pd.read_csv('../data/contracts_clean.csv')
active_core = pd.read_csv('../data/active_core_firms.csv')

print(f'Bids table: {len(bids_table):,} bids')
print(f'Contracts: {len(contracts):,}')
print(f'Active core firms: {len(active_core):,}')
print(f'Unique firms in bids: {bids_table["firm_nif"].nunique():,}')
print(f'Unique contracts: {bids_table["idcontrato"].nunique():,}')

# <font color='#BFD72F' size=6>**2. Binary Bid Matrix**</font> <a class="anchor" id="2-bid-matrix"></a>
[Back to TOC](#toc)

Construct the binary firm-tender matrix **B** where $b_{it} = 1$ if firm $i$ bid on tender $t$.

Paper: *"$b_{it}$ is one if firm $i$ made a bid to tender $t$ and zero otherwise."*

In [ ]:
# Create firm and contract indices
firms = sorted(bids_table['firm_nif'].unique())
contracts_ids = sorted(bids_table['idcontrato'].unique())

firm_idx = {nif: i for i, nif in enumerate(firms)}
contract_idx = {cid: j for j, cid in enumerate(contracts_ids)}

N_firms = len(firms)
N_contracts = len(contracts_ids)

print(f'Bid matrix dimensions: {N_firms} firms × {N_contracts} contracts')

# Build sparse binary matrix
rows = bids_table['firm_nif'].map(firm_idx).values
cols = bids_table['idcontrato'].map(contract_idx).values
data = np.ones(len(bids_table), dtype=np.int8)

B = sparse.csr_matrix((data, (rows, cols)), shape=(N_firms, N_contracts))
print(f'Sparse matrix: {B.nnz:,} non-zero entries')
print(f'Density: {B.nnz / (N_firms * N_contracts):.6f}')

In [ ]:
# Compute firm activity: p_i = fraction of tenders firm i participated in
n_bids_per_firm = np.array(B.sum(axis=1)).flatten()  # n_i for each firm
p_i = n_bids_per_firm / N_contracts  # participation rate

print(f'Bids per firm: min={n_bids_per_firm.min()}, median={np.median(n_bids_per_firm):.0f}, max={n_bids_per_firm.max()}')
print(f'Participation rate: min={p_i.min():.6f}, median={np.median(p_i):.6f}, max={p_i.max():.4f}')

# <font color='#BFD72F' size=6>**3. Centered Jaccard-Tanimoto Coefficient**</font> <a class="anchor" id="3-jaccard"></a>
[Back to TOC](#toc)

**Equation 1 from the paper:**

$J^c_{ij} = \\frac{\\sum_t b_{it} \\cdot b_{jt} - T \\cdot p_i \\cdot p_j}{\\sum_t (b_{it} + b_{jt} - b_{it} \\cdot b_{jt}) - T \\cdot p_i \\cdot p_j}$

Where:
- $b_{it} = 1$ if firm $i$ bid on tender $t$
- $p_i = \\frac{1}{T}\\sum_t b_{it}$ = fraction of tenders firm $i$ participated in
- $T$ = total number of tenders

*"The centered Jaccard coefficient allows us to distinguish between positive and negative associations between firms. We are interested only in associations that are positive and significant."*

In [ ]:
# Step 1: Compute co-occurrence matrix (B @ B.T)
# This gives n_ij = number of tenders both firms i and j bid on
print('Computing co-occurrence matrix (B @ B.T)...')
co_occurrence = B @ B.T  # sparse matrix, shape (N_firms, N_firms)
print(f'Co-occurrence matrix computed. Non-zero pairs: {(co_occurrence.nnz - N_firms) // 2:,}')

# Convert to COO for efficient iteration
co_occ_coo = sparse.triu(co_occurrence, k=1).tocoo()  # upper triangle only

In [ ]:
# Step 2: Compute Centered Jaccard for all pairs with co-occurrence > 0
print(f'Computing Centered Jaccard-Tanimoto for {len(co_occ_coo.data):,} pairs...')

T = N_contracts
edge_data = []  # Will store (i, j, Jc, n_ij)

for idx in range(len(co_occ_coo.data)):
    i = co_occ_coo.row[idx]
    j = co_occ_coo.col[idx]
    n_ij = co_occ_coo.data[idx]  # co-occurrences
    
    n_i = n_bids_per_firm[i]
    n_j = n_bids_per_firm[j]
    
    # Expected co-occurrence under independence
    E_ij = T * p_i[i] * p_i[j]
    
    # Union
    u_ij = n_i + n_j - n_ij
    
    # Centered Jaccard
    denominator = u_ij - E_ij
    if denominator <= 0:
        continue
    
    Jc = (n_ij - E_ij) / denominator
    
    # Keep only positive associations (paper: "we are interested only in positive")
    if Jc > 0:
        edge_data.append((i, j, Jc, n_ij))

print(f'Positive Jc pairs: {len(edge_data):,}')

# <font color='#BFD72F' size=6>**4. Statistical Significance Testing**</font> <a class="anchor" id="4-significance"></a>
[Back to TOC](#toc)

Paper: *"We bootstrap a null distribution of the centered Jaccard coefficient by generating an ensemble of 1000 randomizations. We discard links with p-value ≥ 0.05."*

**Implementation note:** Full bootstrap (1000 iterations) is computationally expensive for large matrices. We use an analytical approximation based on the expected distribution under the null hypothesis of independent bidding behavior.

In [ ]:
# Analytical significance test
# Under null (independent bidding): E[n_ij] = T * p_i * p_j
# Var[n_ij] ≈ T * p_i * p_j * (1 - p_i) * (1 - p_j) (binomial approximation)
# We use a z-test: z = (n_ij - E_ij) / sqrt(Var_ij)
# Additionally, require minimum overlap (n_ij >= 3) for robustness

MIN_OVERLAP = 3  # minimum shared tenders for edge to be considered
ALPHA = 0.05  # significance level

significant_edges = []

for i, j, Jc, n_ij in edge_data:
    # Minimum overlap filter
    if n_ij < MIN_OVERLAP:
        continue
    
    # Expected and variance under null
    E_ij = T * p_i[i] * p_i[j]
    Var_ij = T * p_i[i] * p_i[j] * (1 - p_i[i]) * (1 - p_i[j])
    
    if Var_ij <= 0:
        continue
    
    # Z-score
    z = (n_ij - E_ij) / np.sqrt(Var_ij)
    
    # One-sided p-value (testing Jc > 0)
    p_value = 1 - stats.norm.cdf(z)
    
    if p_value < ALPHA:
        significant_edges.append({
            'firm_i': firms[i],
            'firm_j': firms[j],
            'Jc': Jc,
            'n_shared': n_ij,
            'z_score': z,
            'p_value': p_value
        })

print(f'Significant edges (p < {ALPHA}): {len(significant_edges):,}')
print(f'Rejected {len(edge_data) - len(significant_edges):,} non-significant pairs')

# <font color='#BFD72F' size=6>**5. Network Construction**</font> <a class="anchor" id="5-network"></a>
[Back to TOC](#toc)

Paper: *"We inferred the network as weighted, using the observed Jaccard-Tanimoto coefficient as the edge weight."*

In [ ]:
# Build undirected weighted graph
G = nx.Graph()

# Add nodes with attributes
firm_name_map = dict(zip(bids_table['firm_nif'], bids_table['firm_name']))
for nif in firms:
    idx = firm_idx[nif]
    G.add_node(nif, 
               firm_name=firm_name_map.get(nif, ''),
               n_bids=int(n_bids_per_firm[idx]))

# Add significant edges
for edge in significant_edges:
    G.add_edge(edge['firm_i'], edge['firm_j'],
               weight=edge['Jc'],
               n_shared=edge['n_shared'],
               p_value=edge['p_value'])

# Remove isolated nodes (firms with no significant co-bidding ties)
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.6f}')
print(f'Removed {len(isolated)} isolated nodes')

In [ ]:
# Extract Giant Component
components = sorted(nx.connected_components(G), key=len, reverse=True)
giant = G.subgraph(components[0]).copy()

print(f'\nConnected components: {len(components)}')
print(f'Giant component: {giant.number_of_nodes()} nodes ({100*giant.number_of_nodes()/G.number_of_nodes():.1f}%), {giant.number_of_edges()} edges')
print(f'\nSmaller components (top 5):')
for i, comp in enumerate(components[1:6]):
    print(f'  Component {i+2}: {len(comp)} nodes')

# <font color='#BFD72F' size=6>**6. Summary Statistics**</font> <a class="anchor" id="6-summary"></a>
[Back to TOC](#toc)

Following the paper's Table 1 format.

In [ ]:
# Compute statistics matching Paper Table 1
degrees = [d for _, d in giant.degree()]
clustering = list(nx.clustering(giant).values())

stats_table = {
    'General Network Statistics': '',
    'n Edges (Full Graph)': G.number_of_edges(),
    'n Nodes (Full Graph)': G.number_of_nodes(),
    'n Edges (Giant Comp.)': giant.number_of_edges(),
    'n Nodes (Giant Comp.)': giant.number_of_nodes(),
    'Node-Level Statistics': '',
    'Median Degree': int(np.median(degrees)),
    'Q1 Degree': int(np.percentile(degrees, 25)),
    'Q3 Degree': int(np.percentile(degrees, 75)),
    'Median Clustering Coef.': f'{np.median(clustering):.3f}',
    'Q1 Clustering Coef.': f'{np.percentile(clustering, 25):.3f}',
    'Q3 Clustering Coef.': f'{np.percentile(clustering, 75):.3f}',
}

print('='*50)
print('NETWORK SUMMARY STATISTICS')
print('(Following Sturm et al. 2025, Table 1)')
print('='*50)
for key, val in stats_table.items():
    if val == '':
        print(f'\n--- {key} ---')
    else:
        print(f'{key:30s} {val}')
print('='*50)

# <font color='#BFD72F' size=6>**7. Export**</font> <a class="anchor" id="7-export"></a>
[Back to TOC](#toc)

In [ ]:
# Export network
os.makedirs('../graphs', exist_ok=True)
nx.write_graphml(giant, '../graphs/cobidding_network.graphml')
print(f'Saved: ../graphs/cobidding_network.graphml')

# Export edge list
edges_df = pd.DataFrame(significant_edges)
edges_df.to_csv('../data/cobidding_edges.csv', index=False)
print(f'Saved: ../data/cobidding_edges.csv ({len(edges_df):,} edges)')

# Export node attributes
node_data = []
for node in giant.nodes():
    node_data.append({
        'firm_nif': node,
        'firm_name': giant.nodes[node].get('firm_name', ''),
        'degree': giant.degree(node),
        'weighted_degree': giant.degree(node, weight='weight'),
        'clustering_coef': nx.clustering(giant, node),
        'n_bids': giant.nodes[node].get('n_bids', 0)
    })
nodes_df = pd.DataFrame(node_data)
nodes_df.to_csv('../data/cobidding_nodes.csv', index=False)
print(f'Saved: ../data/cobidding_nodes.csv ({len(nodes_df):,} nodes)')